# 类型校验


除了pydantic字段不匹配会抛出异常，其他三种不进行校验。

# 获取结构化结果方式
## 1. 使用with_structured_output
这种方式是 最新 、 最简洁 的API，直接让模型“理解”你需要的数据结构，并返回解析好的对象。
还可以在with_structured_output方法中传入 `include_raw=True` 参数，表示返回解析前的 `原始AIMessage` ，从而访问令牌用量等元数据。
输出包含了完整的输出响应，包含三个字段
- raw：返回的原始AIMessage。
- parsed：解析后的输出
- parsing_error：解析错误，当前用的是Pydantic，校验，格式不符合schema会导致报错。其它三种方式不符合schema不会导致报错。

In [1]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool  # 从 core 导入，避免 langgraph 版本冲突
from rich import print as rprint
from langchain_qwq import ChatQwen
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="评分（10分制）")
# 设置模型结构化输出
model_with_structure = model.with_structured_output(Movie,include_raw=True)
# 调用模型并获取结构化输出
resp = model_with_structure.invoke("给我介绍下电影《星际穿越》")
print(type(resp))
rprint(resp)

<class 'dict'>


{
    'raw': AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 85,
                'prompt_tokens': 359,
                'total_tokens': 444,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
            },
            'model_provider': 'dashscope',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': None,
            'id': 'chatcmpl-e875bd2e-0b86-9635-8ee1-560fb0c447af',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f17ad-4521-70d2-9530-a8f24497c6ff-0',
        tool_calls=[
            {
                'name': 'Movie',
                'args': {'title': '星际穿越', 'year': 2014, 'director': '克里斯托弗·诺兰', 'rating': 9.4},
                'id': 'call_4b17d628ee7b470fb5923c6f',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 359,
            'output_tokens': 85,
            'total_tokens': 444,
            'input_token_details': {'cache_read': 0},
            'output_token_details': {}
        }
    ),
    'parsed': Movie(title='星际穿越', year=2014, director='克里斯托弗·诺兰', rating=9.4),
    'parsing_error': None
}

## 2. 使用输出解析器

这种方法更传统，依赖于在提示词中明确指示模型输出特定格式的文本，然后使用解析器进行转换。

其流程是： 提示词指导 (引导生成指定类型） → 模型生成文本 → 解析器转换 。